In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)

In [4]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [5]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [6]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1 # using b1 just for fun, it's useless because of BN
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

# Note: I am initializating many of these parameters in non-standard ways
# because sometimes initializating with e.g. all zeros could mask an incorrect
# implementation of the backward pass.

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

4137


In [7]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

# Shapes

In [8]:
# embeddings
emb = C[Xb]
embcat = emb.view(emb.shape[0], -1)

print(Xb.shape)
print(C.shape)
print(emb.shape)
print(embcat.shape)

torch.Size([32, 3])
torch.Size([27, 10])
torch.Size([32, 3, 10])
torch.Size([32, 30])


In [9]:
# linear layer
hprebn = embcat @ W1 + b1

print(W1.shape)
print(hprebn.shape)

torch.Size([30, 64])
torch.Size([32, 64])


In [10]:
# batch norm
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani

bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias


print(bnmeani.shape)
print(bndiff.shape)

print()

print(bndiff2.shape)
print(bnvar.shape)
print(bnvar_inv.shape)
print(bnraw.shape)
print(hpreact.shape)

torch.Size([1, 64])
torch.Size([32, 64])

torch.Size([32, 64])
torch.Size([1, 64])
torch.Size([1, 64])
torch.Size([32, 64])
torch.Size([32, 64])


In [11]:
# non-linearity
h = torch.tanh(hpreact)

print(h.shape)

torch.Size([32, 64])


In [12]:
# linear layer
logits = h @ W2 + b2

print(W2.shape)
print(logits.shape)

torch.Size([64, 27])
torch.Size([32, 27])


In [13]:
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

print(f"{logit_maxes.shape=}")
print(f"{norm_logits.shape=}")
print(f"{counts.shape=}")
print(f"{counts_sum.shape=}")
print(f"{counts_sum_inv.shape=}")
print(f"{probs.shape=}")
print(f"{logprobs.shape=}")
print(f"{loss.shape=}")

logit_maxes.shape=torch.Size([32, 1])
norm_logits.shape=torch.Size([32, 27])
counts.shape=torch.Size([32, 27])
counts_sum.shape=torch.Size([32, 1])
counts_sum_inv.shape=torch.Size([32, 1])
probs.shape=torch.Size([32, 27])
logprobs.shape=torch.Size([32, 27])
loss.shape=torch.Size([])


In [14]:
t = -logprobs[range(n), Yb]
t, t.shape

(tensor([3.9987, 3.1509, 3.5434, 3.2638, 4.1093, 3.5237, 3.2519, 4.0969, 3.2377,
         4.2511, 3.1768, 1.6035, 2.7897, 3.0462, 2.9550, 3.1587, 3.9289, 2.9394,
         3.5756, 3.4201, 2.7678, 2.8630, 4.2871, 3.9590, 3.5161, 2.9400, 3.0620,
         3.8185, 2.8761, 3.5207, 3.2511, 3.2373], grad_fn=<NegBackward0>),
 torch.Size([32]))

In [15]:
tm = t.mean()
tm, tm.shape

(tensor(3.3475, grad_fn=<MeanBackward0>), torch.Size([]))

# Forward pass

In [16]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation
# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact) # hidden layer
# Linear layer 2
logits = h @ W2 + b2 # output layer
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.3475, grad_fn=<NegBackward0>)

# Backward pass

In [17]:
# Exercise 1: backprop through the whole thing manually, 
# backpropagating through exactly all of the variables 
# as they are defined in the forward pass above, one by one

# -----------------
# YOUR CODE HERE :)
# -----------------

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1/n

dprobs = 1.0/probs * dlogprobs

dcounts_sum_inv = (counts*dprobs).sum(1, keepdim=True)

dcounts = counts_sum_inv * dprobs

dcounts_sum = -1/counts_sum**2 * dcounts_sum_inv

dcounts += dcounts_sum.expand(-1, 27) # add contributions

dnorm_logits = norm_logits.exp() * dcounts # counts * dcounts

dlogit_maxes = -dnorm_logits.sum(1, keepdim=True)

dlogits = dnorm_logits.clone()

max_indices = logits.max(1, keepdim=True).indices
dlogits.scatter_add_(1, max_indices, dlogit_maxes)

# ---------------------------------------------------------------------

dh = dlogits @ W2.T

dW2 = h.T @ dlogits

db2 = dlogits.sum(0)

dhpreact = (1.0-h**2) * dh

# ---------------------------------------------------------------------

dbngain = (bnraw * dhpreact).sum(0, keepdim=True)

dbnbias = dhpreact.sum(0, keepdim=True)

dbnraw = bngain * dhpreact

dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)

dbndiff = bnvar_inv * dbnraw # node 1

dbnvar = -0.5 * (bnvar + 1e-5)**(-1.5) * dbnvar_inv

dbndiff2 = ((1 / (n - 1)) * dbnvar)
dbndiff2 = dbndiff2.expand(32, -1)

dbndiff += 2*bndiff * dbndiff2 # node 2

dbnmeani = -dbndiff.sum(0, keepdim=True)

dhprebn = dbndiff.clone() # node 1
dhprebn += ((1/n) * dbnmeani).expand(32,-1) # node 2

# ---------------------------------------------------------------------

dembcat = dhprebn @ W1.T

dW1 = embcat.T @ dhprebn

db1 = dhprebn.sum(0)

# ---------------------------------------------------------------------

demb = dembcat.view(-1, emb.shape[1], emb.shape[2])

dC = torch.zeros_like(C)

for i in range(32):
    for j in range(3):
        dC[Xb[i, j]] += demb[i, j]

cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)

cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)

cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)

cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)

cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)

cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)

cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff:

In [18]:
# Verify that every manual gradient has the same shape
# as the tensor whose gradient it represents

gradient_pairs = [
    ("logprobs",       dlogprobs,       logprobs),
    ("probs",          dprobs,          probs),
    ("counts_sum_inv", dcounts_sum_inv, counts_sum_inv),
    ("counts_sum",     dcounts_sum,     counts_sum),
    ("counts",         dcounts,         counts),
    ("norm_logits",    dnorm_logits,    norm_logits),
    ("logit_maxes",    dlogit_maxes,    logit_maxes),
    ("logits",         dlogits,         logits),

    ("h",              dh,              h),
    ("W2",             dW2,             W2),
    ("b2",             db2,             b2),
    ("hpreact",        dhpreact,        hpreact),

    ("bngain",         dbngain,         bngain),
    ("bnbias",         dbnbias,         bnbias),
    ("bnraw",          dbnraw,          bnraw),
    ("bnvar_inv",      dbnvar_inv,      bnvar_inv),
    ("bnvar",          dbnvar,          bnvar),
    ("bndiff2",        dbndiff2,        bndiff2),
    ("bndiff",         dbndiff,         bndiff),
    ("bnmeani",        dbnmeani,        bnmeani),
    ("hprebn",         dhprebn,         hprebn),

    ("embcat",         dembcat,         embcat),
    ("W1",             dW1,             W1),
    ("b1",             db1,             b1),
    ("emb",            demb,            emb),
    ("C",              dC,              C),
]

for name, manual_grad, tensor in gradient_pairs:
    assert manual_grad.shape == tensor.shape, (
        f"{name}: manual gradient has shape {manual_grad.shape}, "
        f"but tensor has shape {tensor.shape}"
    )

print("All gradient shapes match.")

All gradient shapes match.


In [19]:
# Exercise 2: backprop through cross_entropy but all in one go
# to complete this challenge look at the mathematical expression of the loss,
# take the derivative, simplify the expression, and just write it out

# forward pass

# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

3.3474838733673096 diff: 2.384185791015625e-07


In [20]:
# backward pass

# -----------------
# YOUR CODE HERE :)
dlogits = torch.zeros_like(logits)

for k in range(32): # examples
    s = logits[k].exp().sum()
    for i in range(27): # categories
        dlogits[k,i] += 1.0/32 * logits[k,i].exp() * s**-1
    dlogits[k,Yb[k]] += -1.0/32
    
cmp('logits', dlogits, logits) # I can only get approximate to be true, my maxdiff is 6e-9

logits          | exact: False | approximate: True  | maxdiff: 5.820766091346741e-09


In [21]:
dlogits = torch.zeros_like(logits)

for k in range(32):
    dlogits[k] = logits[k].exp() / logits[k].exp().sum()
    dlogits[k, Yb[k]] -= 1
dlogits/=32

cmp('logits', dlogits, logits)

logits          | exact: False | approximate: True  | maxdiff: 5.587935447692871e-09


In [22]:
dlogits = logits.softmax(dim=1)
dlogits[torch.arange(32), Yb] -= 1
dlogits /= 32

cmp('logits', dlogits, logits)

logits          | exact: False | approximate: True  | maxdiff: 5.587935447692871e-09


In [23]:
# Exercise 3: backprop through batchnorm but all in one go
# to complete this challenge look at the mathematical expression of the output of batchnorm,
# take the derivative w.r.t. its input, simplify the expression, and just write it out
# BatchNorm paper: https://arxiv.org/abs/1502.03167

# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# now:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

max diff: tensor(7.1526e-07, grad_fn=<MaxBackward1>)


In [24]:
# backward pass

# before we had:
# dbnraw = bngain * dhpreact
# dbndiff = bnvar_inv * dbnraw
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
# dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
# dbndiff += (2*bndiff) * dbndiff2
# dhprebn = dbndiff.clone()
# dbnmeani = (-dbndiff).sum(0)
# dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani)

# calculate dhprebn given dhpreact (i.e. backprop through the batchnorm)
# (you'll also need to use some of the variables from the forward pass up above)

# -----------------
# YOUR CODE HERE :)
n = hprebn.shape[0]

term1 = dhpreact
term2 = dhpreact.sum(dim=0, keepdim=True) / n

weighted_sum = (dhpreact * bnraw).sum(
    dim=0,
    keepdim=True
)
term3 = bnraw * weighted_sum / (n - 1)

scale = bngain * bnvar_inv

dhprebn = scale * (term1 - term2 - term3)
# -----------------

cmp('hprebn', dhprebn, hprebn) # I can only get approximate to be true, my maxdiff is 9e-10

hprebn          | exact: False | approximate: True  | maxdiff: 9.313225746154785e-10


$$
\frac{\partial L}{\partial x_j}
=
\sum_{i=1}^{m}
\frac{\partial L}{\partial y_i}
\frac{\partial y_i}{\partial x_j}
$$

$$
\frac{\partial L}{\partial x_{j,c}}
=
\frac{\alpha_c}{\sigma_c}
\left[
\underbrace{g_{j,c}}_{\text{term 1}}
-
\underbrace{\frac{1}{n}\sum_{i=1}^{n}g_{i,c}}_{\text{term 2}}
-
\underbrace{
\frac{\hat{x}_{j,c}}{n-1}
\sum_{i=1}^{n}g_{i,c}\hat{x}_{i,c}
}_{\text{term 3}}
\right]
$$

In [25]:
n = hprebn.shape[0]

dhprebn = bngain*bnvar_inv * (dhpreact - dhpreact.sum(0, keepdim=True)/n - bnraw*(dhpreact*bnraw).sum(0, keepdim=True)/(n-1))
cmp('hprebn', dhprebn, hprebn) 

hprebn          | exact: False | approximate: True  | maxdiff: 9.313225746154785e-10


In [26]:
# hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias

In [27]:
hpreact.shape, hprebn.shape

(torch.Size([32, 64]), torch.Size([32, 64]))

In [28]:
n = n = hprebn.shape[0]
n

32

In [29]:
term1 = dhpreact
term1.shape

torch.Size([32, 64])

In [30]:
term2 = dhpreact.sum(0, keepdim=True)/n
term2.shape

torch.Size([1, 64])

In [31]:
weighted_sum = (dhpreact * bnraw).sum(0, keepdim=True)
weighted_sum.shape

torch.Size([1, 64])

In [32]:
term3 = bnraw * weighted_sum / (n - 1)
term3.shape, bnraw.shape

(torch.Size([32, 64]), torch.Size([32, 64]))

In [33]:
scale = bngain * bnvar_inv
scale.shape

torch.Size([1, 64])

In [34]:
# Exercise 4: putting it all together!
# Train the MLP neural net with your own backward pass

# init
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

# same optimization as last time
max_steps = 200000
batch_size = 32
n = batch_size # convenience
lossi = []

# use this context manager for efficiency once your backward pass is written (TODO)
with torch.no_grad():

    # kick off optimization
    for i in range(max_steps):
    
      # minibatch construct
      ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
      Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
    
      # forward pass
      emb = C[Xb] # embed the characters into vectors
      embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
      # Linear layer
      hprebn = embcat @ W1 + b1 # hidden layer pre-activation
      # BatchNorm layer
      # -------------------------------------------------------------
      bnmean = hprebn.mean(0, keepdim=True)
      bnvar = hprebn.var(0, keepdim=True, unbiased=True)
      bnvar_inv = (bnvar + 1e-5)**-0.5
      bnraw = (hprebn - bnmean) * bnvar_inv
      hpreact = bngain * bnraw + bnbias
      # -------------------------------------------------------------
      # Non-linearity
      h = torch.tanh(hpreact) # hidden layer
      logits = h @ W2 + b2 # output layer
      loss = F.cross_entropy(logits, Yb) # loss function
    
      # backward pass
      for p in parameters:
        p.grad = None
      # loss.backward() # use this for correctness comparisons, delete it later!
    
      # manual backprop! #swole_doge_meme
      # -----------------
      # YOUR CODE HERE :)
      dlogits = logits.softmax(dim=1)
      dlogits[torch.arange(32), Yb] -= 1
      dlogits /= n
    
      dh = dlogits @ W2.T
      dW2 = h.T @ dlogits
      db2 = dlogits.sum(0)
        
      dhpreact = (1.0-h**2) * dh
    
      dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
      dbnbias = dhpreact.sum(0, keepdim=True)
      dhprebn = bngain*bnvar_inv * (dhpreact - dhpreact.sum(0, keepdim=True)/n - bnraw*(dhpreact*bnraw).sum(0, keepdim=True)/(n-1))
    
      dembcat = dhprebn @ W1.T
      dW1 = embcat.T @ dhprebn
      db1 = dhprebn.sum(0)
    
      demb = dembcat.view(-1, emb.shape[1], emb.shape[2])
      dC = torch.zeros_like(C)
      for k in range(32):
        for j in range(3):
            dC[Xb[k, j]] += demb[k, j]
    
      grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
    
      # update
      lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
      for p, grad in zip(parameters, grads):
        # p.data += -lr * p.grad # old way of cheems doge (using PyTorch grad from .backward())
        p.data += -lr * grad # new way of swole doge TODO: enable
    
      # track stats
      if i % 10000 == 0: # print every once in a while
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
      lossi.append(loss.log10().item())
    
      # if i >= 100: # TODO: delete early breaking when you're ready to train the full net
      #  break

12297
      0/ 200000: 3.8108
  10000/ 200000: 2.1692
  20000/ 200000: 2.3920
  30000/ 200000: 2.4456
  40000/ 200000: 1.9572
  50000/ 200000: 2.3816
  60000/ 200000: 2.3585
  70000/ 200000: 2.0489
  80000/ 200000: 2.3283
  90000/ 200000: 2.1649
 100000/ 200000: 2.0119
 110000/ 200000: 2.3550
 120000/ 200000: 1.9884
 130000/ 200000: 2.3740
 140000/ 200000: 2.3568
 150000/ 200000: 2.1142
 160000/ 200000: 2.0057
 170000/ 200000: 1.8387
 180000/ 200000: 2.0239
 190000/ 200000: 1.8109


In [35]:
# useful for checking your gradients
#for p,g in zip(parameters, grads):
#  cmp(str(tuple(p.shape)), g, p)

In [36]:
# calibrate the batch norm at the end of training

with torch.no_grad():
  # pass the training set through
  emb = C[Xtr]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 + b1
  # measure the mean/std over the entire training set
  bnmean = hpreact.mean(0, keepdim=True)
  bnvar = hpreact.var(0, keepdim=True, unbiased=True)


In [37]:
# evaluate train and val loss

@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 + b1
  hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.0707335472106934
val 2.1077828407287598


In [38]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # forward pass
      emb = C[torch.tensor([context])] # (1,block_size,d)      
      embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
      hpreact = embcat @ W1 + b1
      hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
      h = torch.tanh(hpreact) # (N, n_hidden)
      logits = h @ W2 + b2 # (N, vocab_size)
      # sample
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

carlah.
ambrie.
khyrmoni.
taty.
skanden.
jazonte.
den.
arci.
aqui.
nellara.
chaiivon.
leigh.
ham.
joce.
quinn.
shon.
marianni.
wavero.
dearisi.
jace.
